# streamlit

* ``Streamlit`` is an open-source Python framework for data scientists and AI/ML engineers to deliver interactive data apps – in only a few lines of code.
* Read more from ``Streamlit``'s documentation [streamlitDoc](https://docs.streamlit.io/)

## Installation

In [ ]:
!pip install streamlit

# Imports need packages

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Create the Streamlit page header

In [ ]:
page_header_code = 'import streamlit as st\n\nst.set_page_config(\n    page_title="KNN Classifier",\n    page_icon="📊",\n    layout="wide"\n)\n\nst.title("K-Nearest Neighbors Classifier")'
print(page_header_code)

import streamlit as st

st.set_page_config(
    page_title="KNN Classifier",
    page_icon="📊",
    layout="wide"
)

st.title("K-Nearest Neighbors Classifier")


# Cache data loading

* ``Streamlit`` reruns the script when a user changes a widget.
* `` @st.cache_data`` avoids unnecessarily rereading the Excel file every time.

In [ ]:
streamlit_loader_code = '@st.cache_data\ndef load_data():\n    data = pd.read_excel("dataset.xls")\n    if "row" in data.columns:\n        data = data.drop(columns=["row"])\n    data = pd.get_dummies(data, columns=["famhist"], drop_first=True, dtype=float)\n    return data'
print(streamlit_loader_code)

@st.cache_data
def load_data():
    data = pd.read_excel("dataset.xls")
    if "row" in data.columns:
        data = data.drop(columns=["row"])
    data = pd.get_dummies(data, columns=["famhist"], drop_first=True, dtype=float)
    return data


# Organize the app into four tabs

In [ ]:
tabs_code = 'tab1, tab2, tab3, tab4 = st.tabs(\n    ["Explore Data", "Choose K", "Evaluate Model", "Make a Prediction"]\n)'
print(tabs_code)

tab1, tab2, tab3, tab4 = st.tabs(
    ["Explore Data", "Choose K", "Evaluate Model", "Make a Prediction"]
)


# Replace hard-coded patient values with interactive widgets

The full app uses ``st.number_input``, ``st.selectbox``, and a ``Predict`` button to collect a new observation.

In [ ]:
interactive_example = 'sbp = st.number_input("Systolic blood pressure", 50.0, 250.0, 130.0)\nfamhist = st.selectbox("Family history of heart disease", ["Absent", "Present"])\n\nif st.button("Predict"):\n    # Build new_patient and call final_model.predict(new_patient)\n    pass'
print(interactive_example)

sbp = st.number_input("Systolic blood pressure", 50.0, 250.0, 130.0)
famhist = st.selectbox("Family history of heart disease", ["Absent", "Present"])

if st.button("Predict"):
    # Build new_patient and call final_model.predict(new_patient)
    pass


# Generate the complete Streamlit app

In [ ]:
from pathlib import Path
app_code = 'import streamlit as st\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom sklearn.model_selection import train_test_split, cross_val_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.metrics import (\n    accuracy_score,\n    confusion_matrix,\n    classification_report\n)\n\n# -----------------------------\n# Page title\n# -----------------------------\n\nst.set_page_config(\n    page_title="KNN Classifier",\n    page_icon="📊",\n    layout="wide"\n)\n\nst.title("K-Nearest Neighbors Classifier")\nst.write(\n    "This app demonstrates how the K-nearest-neighbors algorithm "\n    "can be used to classify heart-disease observations."\n)\n\n# -----------------------------\n# Load and prepare the data\n# -----------------------------\n\n@st.cache_data\ndef load_data():\n    data = pd.read_excel("dataset.xls")\n\n    # Remove the row-number column\n    if "row" in data.columns:\n        data = data.drop(columns=["row"])\n\n    # Convert categorical variables to numerical variables\n    data = pd.get_dummies(\n        data,\n        columns=["famhist"],\n        drop_first=True,\n        dtype=float\n    )\n\n    return data\n\n\ndf = load_data()\n\nX = df.drop(columns=["chd"])\ny = df["chd"]\n\n# Split the data before standardization\nX_train, X_test, y_train, y_test = train_test_split(\n    X,\n    y,\n    test_size=0.30,\n    random_state=101,\n    stratify=y\n)\n\n# -----------------------------\n# App tabs\n# -----------------------------\n\ntab1, tab2, tab3, tab4 = st.tabs(\n    ["Explore Data", "Choose K", "Evaluate Model", "Make a Prediction"]\n)\n\n# -----------------------------\n# Tab 1: Explore the data\n# -----------------------------\n\nwith tab1:\n    st.header("Explore the Data")\n\n    st.write("The first five observations are:")\n\n    # Display original-style data\n    st.dataframe(df.head())\n\n    col1, col2, col3 = st.columns(3)\n\n    with col1:\n        st.metric("Number of observations", df.shape[0])\n\n    with col2:\n        st.metric("Number of predictors", X.shape[1])\n\n    with col3:\n        st.metric("Number of classes", y.nunique())\n\n    st.subheader("Class distribution")\n\n    class_counts = y.value_counts().sort_index()\n    class_counts.index = ["No heart disease", "Heart disease"]\n\n    st.bar_chart(class_counts)\n\n    st.write(\n        """\n        The target variable `chd` is coded as follows:\n\n        - `0`: No coronary heart disease\n        - `1`: Coronary heart disease\n        """\n    )\n\n# -----------------------------\n# Tab 2: Choose K\n# -----------------------------\n\nwith tab2:\n    st.header("Choose the Number of Neighbors")\n\n    st.write(\n        """\n        The value of K determines how many nearby observations are used\n        to classify a new observation. Cross-validation is used to compare\n        different values of K using the training data.\n        """\n    )\n\n    k_values = range(1, 40)\n    validation_errors = []\n\n    for k_value in k_values:\n        validation_model = Pipeline([\n            ("scaler", StandardScaler()),\n            ("knn", KNeighborsClassifier(n_neighbors=k_value))\n        ])\n\n        scores = cross_val_score(\n            validation_model,\n            X_train,\n            y_train,\n            cv=5,\n            scoring="accuracy"\n        )\n\n        validation_errors.append(1 - scores.mean())\n\n    best_k = list(k_values)[validation_errors.index(min(validation_errors))]\n\n    st.success(\n        f"The value of K with the smallest cross-validation error is {best_k}."\n    )\n\n    fig, ax = plt.subplots()\n\n    ax.plot(\n        list(k_values),\n        validation_errors,\n        marker="o",\n        linestyle="--"\n    )\n\n    ax.set_xlabel("Number of neighbors, K")\n    ax.set_ylabel("Cross-validation error rate")\n    ax.set_title("Cross-validation Error Rate versus K")\n\n    st.pyplot(fig)\n\n    st.write(\n        """\n        Smaller values of K can produce a flexible model that is sensitive\n        to individual observations. Larger values of K produce a smoother\n        decision rule.\n        """\n    )\n\n# -----------------------------\n# Tab 3: Evaluate the model\n# -----------------------------\n\nwith tab3:\n    st.header("Evaluate the KNN Model")\n\n    selected_k = st.slider(\n        "Select a value of K",\n        min_value=1,\n        max_value=39,\n        value=best_k\n    )\n\n    model = Pipeline([\n        ("scaler", StandardScaler()),\n        ("knn", KNeighborsClassifier(n_neighbors=selected_k))\n    ])\n\n    model.fit(X_train, y_train)\n    predictions = model.predict(X_test)\n\n    accuracy = accuracy_score(y_test, predictions)\n    error_rate = 1 - accuracy\n\n    col1, col2 = st.columns(2)\n\n    with col1:\n        st.metric("Test accuracy", f"{accuracy:.2%}")\n\n    with col2:\n        st.metric("Test error rate", f"{error_rate:.2%}")\n\n    st.subheader("Confusion Matrix")\n\n    matrix = confusion_matrix(y_test, predictions)\n\n    matrix_df = pd.DataFrame(\n        matrix,\n        index=["Actual: No CHD", "Actual: CHD"],\n        columns=["Predicted: No CHD", "Predicted: CHD"]\n    )\n\n    st.dataframe(matrix_df)\n\n    st.subheader("Classification Report")\n\n    report = classification_report(\n        y_test,\n        predictions,\n        target_names=["No CHD", "CHD"],\n        output_dict=True\n    )\n\n    report_df = pd.DataFrame(report).transpose()\n    st.dataframe(report_df.round(3))\n\n# -----------------------------\n# Tab 4: Make a prediction\n# -----------------------------\n\nwith tab4:\n    st.header("Predict for a New Patient")\n\n    st.write(\n        """\n        Enter values for a new observation. The prediction is for educational\n        purposes only and is not a medical diagnosis.\n        """\n    )\n\n    col1, col2 = st.columns(2)\n\n    with col1:\n        sbp = st.number_input(\n            "Systolic blood pressure",\n            min_value=50.0,\n            max_value=250.0,\n            value=130.0\n        )\n\n        tobacco = st.number_input(\n            "Tobacco consumption",\n            min_value=0.0,\n            max_value=50.0,\n            value=3.0\n        )\n\n        ldl = st.number_input(\n            "LDL cholesterol",\n            min_value=0.0,\n            max_value=20.0,\n            value=4.0\n        )\n\n        adiposity = st.number_input(\n            "Adiposity",\n            min_value=0.0,\n            max_value=50.0,\n            value=25.0\n        )\n\n        famhist = st.selectbox(\n            "Family history of heart disease",\n            ["Absent", "Present"]\n        )\n\n    with col2:\n        typea = st.number_input(\n            "Type-A behavior score",\n            min_value=0.0,\n            max_value=100.0,\n            value=50.0\n        )\n\n        obesity = st.number_input(\n            "Obesity",\n            min_value=0.0,\n            max_value=50.0,\n            value=25.0\n        )\n\n        alcohol = st.number_input(\n            "Alcohol consumption",\n            min_value=0.0,\n            max_value=100.0,\n            value=10.0\n        )\n\n        age = st.number_input(\n            "Age",\n            min_value=1.0,\n            max_value=120.0,\n            value=50.0\n        )\n\n    if st.button("Predict"):\n        new_patient = pd.DataFrame({\n            "sbp": [sbp],\n            "tobacco": [tobacco],\n            "ldl": [ldl],\n            "adiposity": [adiposity],\n            "typea": [typea],\n            "obesity": [obesity],\n            "alcohol": [alcohol],\n            "age": [age],\n            "famhist_Present": [\n                1.0 if famhist == "Present" else 0.0\n            ]\n        })\n\n        # Ensure the new observation has exactly the same columns as X\n        new_patient = new_patient.reindex(\n            columns=X.columns,\n            fill_value=0\n        )\n\n        final_model = Pipeline([\n            ("scaler", StandardScaler()),\n            ("knn", KNeighborsClassifier(n_neighbors=best_k))\n        ])\n\n        final_model.fit(X_train, y_train)\n\n        prediction = final_model.predict(new_patient)[0]\n        probabilities = final_model.predict_proba(new_patient)[0]\n\n        if prediction == 1:\n            st.warning(\n                "The model predicts the presence of coronary heart disease."\n            )\n        else:\n            st.success(\n                "The model predicts the absence of coronary heart disease."\n            )\n\n        probability_df = pd.DataFrame({\n            "Class": ["No CHD", "CHD"],\n            "Predicted probability": probabilities\n        })\n\n        st.dataframe(probability_df)'
output_app = Path("app_from_notebook.py")
output_app.write_text(app_code, encoding="utf-8")
print("Created:", output_app.resolve())
print("Lines:", len(app_code.splitlines()))

Created: /Users/012767599/Desktop/SJSUCourses/Math251/Fall2026/ML Labs/app_from_notebook.py
Lines: 359


# Validate the generated app

Compile it before launching. No output other than the success message means the syntax is valid.

In [ ]:
import py_compile
py_compile.compile("app_from_notebook.py", doraise=True)
print("Syntax check passed.")

Syntax check passed.


# Run the app

In [ ]:
pwd

'/Users/012767599/Desktop/SJSUCourses/Math251/Fall2026/ML Labs'

In [ ]:
ls

__pycache__/
app_from_notebook.py
app.py
dataset.xls
K Nearest Neighbors with Python-lab1.ipynb
K Nearest Neighbors with Python-streamlit-lab.ipynb
knn_mini_app/


In [ ]:
# before runing the app make sure the data is in this above pass
# ``dataset.xls`` not in the current  and I added it 

In [ ]:
ls

__pycache__/
app_from_notebook.py
dataset.xls
K Nearest Neighbors with Python-lab1.ipynb
K Nearest Neighbors with Python-streamlit-lab.ipynb
knn_mini_app/


In [2]:
!streamlit run app.py

2026-09-30 14:33:22.240 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.250.1.217:8501

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
^C
  Stopping...


In [ ]:
#!streamlit run app_from_notebook.py

# Putting to github

1.  create a local folder for example in your class folder called : ``knn_mini_app`` and put inside the following files
    *  ``app.py`` --> python file

    *  ``README.md`` ---> R markdown file
  
    *  ``datasets.xls``
  
    *  ``.gitignore``
  
    *  ``requirements.txt``



Copy the following inside ``README.md``

# KNN Heart Disease Classification App

This Streamlit application demonstrates the K-Nearest Neighbors (KNN)
classification algorithm using a coronary heart disease dataset.

## Features

- Explore the dataset
- Select K using cross-validation
- Evaluate the KNN classifier
- Display accuracy and confusion matrix
- Display a classification report
- Predict CHD for a new observation

## Run locally

Install the required packages:

```bash
pip install -r requirements.txt

Copy the following inside ``.gitignore``

# Jupyter
.ipynb_checkpoints/

# Python
__pycache__/
*.pyc

# macOS
.DS_Store

# Virtual environments
venv/
.venv/

# Environment variables
.env

Copy the following inside ``requirements.txt``

streamlit
pandas
matplotlib
scikit-learn
xlrd
openpyxl

2. Go to GitHub and create a new repository. Call it for exmple ``knn_mini_app`` but you can call it any name you like
3. From Terminal on your Mac, navigate to the project folder and then
   * Then initialize Git: ``git init``
   * Add all files: ``git add .``
   * Commit them: ``git commit -m "Initial KNN Streamlit app"``
4. Connect your local folder to the GitHub repository. GitHub will give you a repository URL. The commands will look similar to:
   * ``git branch -M main``
   * ``git remote add origin https://github.com/YOUR-USERNAME/knn_mini_app.git``
   *  ``git push -u origin main``

   Replace YOUR-USERNAME with your GitHub username.
5. After pushing, refresh the GitHub repository page. You should see the following files:
    *  ``app.py``
    * ``dataset.xls``
    * ``requirements.txt``
    * ``README.md``
6. Whenever you modify the app later, upload the changes with:
    * ``git add .``
    * ``git commit -m "Update Streamlit app"``
    * ``git push``